# v4: a new, harder gold set, then fine-tune v2 on the mistakes it finds

v2/v3's gold (200 random captchas, labeled by confirming model pre-labels) is down to 3 misses and can no longer tell
models apart; its labels may also lean towards the model that pre-labeled them. This round:

1. **Hardness**: every CNN in `READERS` reads every captcha → `../cache/hardness.csv`.
2. **Pick gold**: 500 captchas drawn mostly where the readers disagree or are unsure, stratified so a weighted score
   still estimates accuracy on all captchas, plus the old 200 re-checked blind → `../data/gold_v2_sheet.csv` and a
   PDF (15 per page, numbered). You answer the PDF as `serial text` lines.
3. **Gold**: your answers → `../data/gold_v2.csv`; score every model on it (hard-gold and weighted accuracy).
4. **Review** non-gold mistakes, **fine-tune** v2 (the shipped model's base) on all reviews, compare on the new gold.


## 1. Setup and paths

In [ ]:
import csv, json, math, os, random, sys, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sys.path.insert(0, "../scripts")
import prep   # ink(): red-letter mask, Vision helpers; shared with scripts/label_vision.py

# images: ../data/captchas.zip (100,000 captchas), read with prep.open_captcha(id); no loose PNGs
GOLD_CSV = "../data/gold_v2.csv"            # new gold: your answers to the sheet (empty until then); never trained on
OLD_GOLD_CSV = "../data/gold.csv"            # v2/v3's 200 gold, re-checked blind inside the new sheet
REVIEWS = "../data/reviews.json"             # your typed answers (review.py adds to it)
VISION_LABELS = "../data/labels_vision.csv"  # Apple Vision's read of every captcha (scripts/label_vision.py)
TEACHER = "../models/captcha_cnn_v2_ft.pt"   # the shipped model (extension 5.0): fixes alphabet and length
READERS = {                                  # hardness.py: where these disagree, a read is likely wrong
    "v2": "../models/captcha_cnn_v2.pt", "v2_ft": "../models/captcha_cnn_v2_ft.pt",
    "v3": "../models/captcha_cnn_v3.pt", "v3_ft": "../models/captcha_cnn_v3_ft.pt",
}
HARDNESS_CSV = "../cache/hardness.csv"       # every reader's read + confidence of every captcha (regenerable)
SHEET_CSV = "../data/gold_v2_sheet.csv"      # serial -> captcha id + stratum of the gold sheet (pick_gold.py)
SHEET_PDF = "../cache/gold_v2_sheet.pdf"     # the sheet you answer (regenerable from pick_gold.py)
LABELS_CSV = "../data/labels_v3.csv"         # v2_ft's reads: the ordinary captchas the fine-tune keeps as anchors
CACHE = "../cache/ink_images.npz"            # cleaned images, so re-runs skip reading 100k PNGs
MODEL_PATH = "../models/captcha_cnn_v2.pt"   # the baseline fine-tuned here (not retrained)
FT_PATH = "../models/captcha_cnn_v4_ft.pt"   # v2 fine-tuned on all reviews
os.makedirs("../models", exist_ok=True)
os.makedirs("../cache", exist_ok=True)

SEED = 0
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

_teacher = torch.load(TEACHER, map_location="cpu")
ALPHABET, LENGTH = _teacher["alphabet"], _teacher["length"]   # same classes as v1, v2 and the extension
assert (ALPHABET, LENGTH) == (prep.ALPHABET, prep.LENGTH)
CHAR_TO_IDX = {c: i for i, c in enumerate(ALPHABET)}
print("device:", DEVICE, "| alphabet:", ALPHABET)

## 2. Load all images (red letters only, cached after the first run, ~1–2 min), Vision reads and gold

In [ ]:
ALL_IDS = np.array(prep.captcha_ids())

cached = np.load(CACHE) if os.path.exists(CACHE) else None
if cached is not None and np.array_equal(cached["ids"], ALL_IDS):
    imgs = cached["imgs"]
else:
    t = time.time()
    with ThreadPoolExecutor(8) as ex:
        imgs = np.stack(list(ex.map(lambda i: prep.ink(prep.open_captcha(i)), ALL_IDS)))
    np.savez(CACHE, ids=ALL_IDS, imgs=imgs)
    print(f"read {len(imgs)} images in {time.time() - t:.0f}s")
H, W = imgs.shape[1:]
X = torch.from_numpy(imgs).to(DEVICE)   # uint8, all images on the GPU (~0.7 GB)
row_of = {i: k for k, i in enumerate(ALL_IDS)}

VISION = {}   # id -> Vision's read, "" where it gave no valid 5-character answer
if os.path.exists(VISION_LABELS):
    with open(VISION_LABELS) as f:
        VISION = {r["id"]: r["vision"] for r in csv.DictReader(f)}
GOLD = {}     # id -> {label, model, vision, checked}; filled by audit_gold.py
if os.path.exists(GOLD_CSV):
    with open(GOLD_CSV) as f:
        GOLD = {r["id"]: r for r in csv.DictReader(f)}
print("images:", imgs.shape, f"| Vision reads: {len(VISION)}/{len(ALL_IDS)}",
      f"| gold: {len(GOLD)} ({sum(bool(g['checked']) for g in GOLD.values())} checked)")

## 3. Model definition and helpers (same network as v1)

In [ ]:
class CaptchaCNN(nn.Module):
    def __init__(self, n_classes, length=LENGTH, widths=(32, 64, 128, 192), dropout=0.3):
        super().__init__()
        layers, c_in = [], 1
        for c in widths:   # each block halves H and W: 45x150 -> 2x9
            layers += [
                nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
            c_in = c
        self.features = nn.Sequential(*layers)
        n_cols = W // 2 ** len(widths)   # 9 columns left, kept apart so heads know left from right
        self.shared = nn.Sequential(nn.Flatten(), nn.Dropout(dropout),
                                    nn.Linear(c_in * n_cols, 512), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.heads = nn.ModuleList(nn.Linear(512, n_classes) for _ in range(length))

    def forward(self, x):
        z = self.features(x).mean(dim=2)   # average over height -> (B, C, columns)
        z = self.shared(z)
        return torch.stack([head(z) for head in self.heads], dim=1)   # (B, LENGTH, n_classes)

def to_input(x):
    """uint8 (B, H, W) -> float (B, 1, H, W) with ink = 1, paper = 0."""
    return (1 - x.float() / 255).unsqueeze(1)

def load_model(path):
    ckpt = torch.load(path, map_location=DEVICE)
    m = CaptchaCNN(len(ckpt["alphabet"]), ckpt["length"], ckpt["widths"], ckpt["config"]["dropout"]).to(DEVICE)
    m.load_state_dict(ckpt["state_dict"])
    return m.eval(), ckpt

@torch.no_grad()
def model_reads(m, ids, batch_size=2048, desc="model reads"):
    """Model reads + weakest-character confidence for a list of image ids."""
    m.eval()
    reads, confs = [], []
    for k in tqdm(range(0, len(ids), batch_size), desc=desc, leave=False):
        x = to_input(X[torch.tensor([row_of[i] for i in ids[k:k + batch_size]]).to(DEVICE)])
        conf, idx = m(x).softmax(-1).max(-1)
        reads += ["".join(ALPHABET[j] for j in p) for p in idx.cpu().tolist()]
        confs += conf.min(-1).values.cpu().tolist()
    return reads, confs

print(f"{sum(p.numel() for p in CaptchaCNN(len(ALPHABET)).parameters()) / 1e6:.2f}M parameters")

def show(i):
    """The original captcha, big enough to read (reviews and gold checks)."""
    plt.figure(figsize=(7, 2.1))
    plt.imshow(prep.open_captcha(i).convert("RGB"))
    plt.axis("off")
    plt.show()

## 4. Hardness: every reader reads every captcha (~2 min)


In [ ]:
%run -i ../scripts/hardness.py


## 5. Pick the new gold and write the answer sheet (PDF, 15 per page)
Strata and how many to draw from each: `CONF_LOW`, `ALLOC` at the top of `../scripts/pick_gold.py`.
Re-running redraws the same captchas (fixed seed) as long as `hardness.csv` is unchanged.


In [ ]:
%run -i ../scripts/pick_gold.py
